In [1]:
# ==========================================
# ПАРСЕР РАЗДЕЛА СТАТЕЙ В ОДИН БОЛЬШОЙ КОРПУС
# ==========================================
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time
import json
from pathlib import Path

In [3]:
# ==========================================
# НАСТРОЙКИ
# ==========================================
START_URL = "https://physics42.ru/tutorials/metamaterialy/"
BASE_DOMAIN = "physics42.ru"

OUTPUT_DIR = Path("parsed_metamaterials")
OUTPUT_DIR.mkdir(exist_ok=True)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

REQUEST_DELAY = 1.0
TIMEOUT = 20

In [5]:
# ==========================================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ==========================================
def get_soup(url: str):
    """Загружает страницу и возвращает BeautifulSoup."""
    response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


def normalize_url(url: str) -> str:
    """Убирает якоря и хвостовые слеши для единообразия."""
    url = url.split("#")[0].strip()
    if url.endswith("/") and len(url) > len("https://"):
        url = url[:-1]
    return url


def is_internal(url: str) -> bool:
    """Проверяет, что ссылка принадлежит нужному домену."""
    parsed = urlparse(url)
    return BASE_DOMAIN in parsed.netloc


def clean_text(text: str) -> str:
    """Чистит лишние пробелы и пустые строки."""
    lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in lines if line]
    return "\n".join(lines)

In [7]:
# ==========================================
# СБОР ССЫЛОК СО СТАРТОВОЙ СТРАНИЦЫ
# ==========================================
def collect_article_links(start_url: str):
    """
    Собирает ссылки на статьи из основного списка на странице раздела.
    Берём ссылки только внутри текущего раздела tutorials/metamaterialy.
    """
    soup = get_soup(start_url)
    links = set()

    for a in soup.select("a[href]"):
        href = a.get("href", "").strip()
        if not href:
            continue

        full_url = urljoin(start_url, href)
        full_url = normalize_url(full_url)

        if not is_internal(full_url):
            continue

        # Ограничиваемся только нужным разделом
        if "/tutorials/metamaterialy/" not in full_url:
            continue

        # Исключаем саму стартовую страницу как статью
        if full_url == normalize_url(start_url):
            continue

        links.add(full_url)

    return sorted(links)

In [ ]:
# ==========================================
# ИЗВЛЕЧЕНИЕ ОСНОВНОГО ТЕКСТА СТРАНИЦЫ
# ==========================================
def extract_article_data(url: str):
    """
    Пытается вытащить заголовок и основной текст статьи.
    Сначала ищет <article>, если нет — берёт наиболее вероятный контейнер.
    """
    soup = get_soup(url)

    # Удаляем ненужные блоки
    for tag in soup(["script", "style", "noscript", "iframe", "footer", "header"]):
        tag.decompose()

    article_block = soup.find("article")

    if article_block is None:
        # Попытка взять основной контент по типичным контейнерам
        candidates = (
            soup.select("main"),
            soup.select(".content"),
            soup.select(".entry-content"),
            soup.select(".post-content"),
            soup.select(".article"),
            soup.select(".container"),
        )

        chosen = None
        for candidate_list in candidates:
            if candidate_list:
                # Берём самый длинный по тексту элемент
                chosen = max(candidate_list, key=lambda x: len(x.get_text(" ", strip=True)))
                break

        article_block = chosen if chosen is not None else soup.body

    # Заголовок
    title_tag = article_block.find(["h1", "h2"]) or soup.find("h1") or soup.find("title")
    title = title_tag.get_text(" ", strip=True) if title_tag else "Без названия"

    # Удалим возможные боковые меню внутри контейнера
    for bad in article_block.select("aside, nav, .sidebar, .menu, .widget"):
        bad.decompose()

    # Собираем текст по абзацам и заголовкам
    parts = []
    for el in article_block.find_all(["h1", "h2", "h3", "h4", "p", "li"]):
        text = el.get_text(" ", strip=True)
        if text and len(text) > 1:
            parts.append(text)

    text = clean_text("\n".join(parts))

    return {
        "url": url,
        "title": title,
        "text": text
    }

In [ ]:
# ==========================================
# ОСНОВНАЯ ЛОГИКА ПАРСИНГА
# ==========================================
def build_corpus(start_url: str):
    links = collect_article_links(start_url)

    print(f"Найдено ссылок: {len(links)}")

    documents = []
    corpus_parts = []

    for i, link in enumerate(links, start=1):
        try:
            print(f"[{i}/{len(links)}] Парсинг: {link}")
            data = extract_article_data(link)

            if data["text"].strip():
                documents.append(data)

                corpus_parts.append(f"=== {data['title']} ===")
                corpus_parts.append(f"URL: {data['url']}")
                corpus_parts.append(data["text"])
                corpus_parts.append("")

            time.sleep(REQUEST_DELAY)

        except Exception as e:
            print(f"Ошибка при обработке {link}: {e}")

    full_corpus = "\n\n".join(corpus_parts).strip()

    return documents, full_corpus

In [ ]:
# ==========================================
# СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
# ==========================================
def save_results(documents, full_corpus, output_dir: Path):
    json_path = output_dir / "metamaterials_articles.json"
    txt_path = output_dir / "metamaterials_corpus.txt"
    urls_path = output_dir / "metamaterials_urls.txt"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(documents, f, ensure_ascii=False, indent=2)

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(full_corpus)

    with open(urls_path, "w", encoding="utf-8") as f:
        for doc in documents:
            f.write(doc["url"] + "\n")

    print(f"\nСохранено:")
    print(f"- JSON: {json_path}")
    print(f"- TXT:  {txt_path}")
    print(f"- URLS: {urls_path}")

In [ ]:
# ==========================================
# ЗАПУСК
# ==========================================

documents, full_corpus = build_corpus(START_URL)
save_results(documents, full_corpus, OUTPUT_DIR)

print("\nГотово.")
print(f"Количество статей: {len(documents)}")
print(f"Размер корпуса: {len(full_corpus)} символов")

Найдено ссылок: 130
[1/130] Парсинг: https://physics42.ru/tutorials/metamaterialy/aktivnaya-maskirovka-i-adaptivnye-sistemy
[2/130] Парсинг: https://physics42.ru/tutorials/metamaterialy/aktivnye-i-nastraivaemye-metapoverkhnosti
[3/130] Парсинг: https://physics42.ru/tutorials/metamaterialy/akusticheskaya-maskirovka-i-zvukopogloshchenie
[4/130] Парсинг: https://physics42.ru/tutorials/metamaterialy/analiticheskie-metody-rascheta-rezonansnykh-chastot
[5/130] Парсинг: https://physics42.ru/tutorials/metamaterialy/anizotropiya-i-bianizotropiya
[6/130] Парсинг: https://physics42.ru/tutorials/metamaterialy/antenny-na-osnove-metamaterialov
[7/130] Парсинг: https://physics42.ru/tutorials/metamaterialy/aukseticheskie-materialy-s-otritsatelnym-koeffitsientom-puassona
[8/130] Парсинг: https://physics42.ru/tutorials/metamaterialy/besprovodnaya-peredacha-energii-cherez-metamaterialy
[9/130] Парсинг: https://physics42.ru/tutorials/metamaterialy/bezopasnost-i-biosovmestimost-metamaterialov
[10/130] Парс